## 13.02 近似训练


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

import math


### 练习 13.2.1

**题目：** 如何在负采样中对噪声词进行采样？

**解答：** 负采样要求从预定义分布 $P(w)$ 中采样 $K$ 个**不在该上下文窗口内**的噪声词，$P(w)$ 的典型取法是词频的 3/4 次方归一化：

$$P(w_i) = \frac{f(w_i)^{0.75}}{\sum_{j} f(w_j)^{0.75}},$$

其中 $f(w_i)$ 是词 $w_i$ 的频次。取 0.75 次方可以压低高频词（如 “the”）被过度采样的概率、抬高低频词的采样概率，使负样本分布更均匀、训练更稳定。

具体采样可用两种方法：

- **轮盘赌（roulette wheel）**：把 $[0,1]$ 按 $P(w_i)$ 分成区间，生成均匀随机数 $r$，落入哪个区间就采样哪个词；
- **二元搜索（binary search）**：预先计算累积分布函数（CDF），用二分查找定位 $r$ 对应的词，复杂度从 $O(|\mathcal{V}|)$ 降到 $O(\log |\mathcal{V}|)$。

主 notebook 13.3 节 `RandomGenerator` 的实现即等价于“轮盘赌 + 缓存”，每轮缓存 $k$ 个采样结果以减少开销。


### 练习 13.2.2

**题目：** 验证式 (13.2.4) 是否有效。

**解答：** (13.2.4) 是层序 softmax 的归一化性质：

$$\sum_{w \in \mathcal{V}} P(w \mid w_c) = 1.$$

层序 softmax 用一棵二叉树表示词表，每个内部节点对应一次二分类（sigmoid）。对任一内部节点，其两个子分支的概率满足

$$\sigma(x) + \sigma(-x) = \frac{1}{1+e^{-x}} + \frac{1}{1+e^{x}} = 1,$$

即左走、右走的概率之和恒为 1。从根节点出发，每层的概率在各节点内部归一，因此整棵树上所有叶子（词）的条件概率之和为 1，式 (13.2.4) 成立。


### 练习 13.2.3

**题目：** 如何分别使用负采样和分层 softmax 训练连续词袋模型？

**解答：**

**负采样版**：

1. 对每个（上下文词元集合，中心词）样本，按 $P(w)$ 采样 $K$ 个噪声词，构造 $K+1$ 个二分类样本（1 个正例 + $K$ 个负例）；
2. 中心词向量 $\mathbf{v}_c$ 用上下文词向量之和（或均值）代替：$\mathbf{v}_c = \sum_{i} \mathbf{v}_{o_i}$，其中 $o_i$ 遍历上下文窗口内的词；
3. 用二元交叉熵损失（等价于 13.4 节的 `SigmoidBCELoss`）做梯度下降更新中心词与上下文词嵌入。

**层序 softmax 版**：

1. 为词表构造一棵 Huffman 二叉树（高频词路径短，低频词路径长）；
2. 对每个词，用从根到叶路径上各内部节点的 sigmoid 概率之积近似 $P(w \mid \mathbf{v}_c)$；
3. 损失为负对数似然 $L = -\log P(w \mid \mathbf{v}_c)$，沿路径逐节点求导更新上下文词向量与内部节点参数。

两者都避免了全词表 softmax 的 $O(|\mathcal{V}|)$ 归一化开销。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
